# [첫번째 제출]

"Polynomial Feature로 변수 상호작용을 추가하고,
Linear + Ridge 앙상블 후 반올림으로 RMSE를 낮춤"

###✳️ import / 라이브러리 호출

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from google.colab import files
upload = files.upload()

# 👉 불필요한 경고 메시지 제거 (코드 가독성 향상)
warnings.filterwarnings('ignore')

###✳️ Fixed RandomSeed / 랜덤시드 고정

In [ ]:
# 👉 실행할 때마다 결과가 달라지는 것을 방지 (재현성 확보)
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

###✳️ Data Load / 데이터 불러오기

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')

###✳️ Feature & Target Split / 독립변수, 종속변수로 나누기

In [ ]:
# 👉 모델 학습을 위해 입력(X)과 정답(y) 분리
# 👉 ID는 의미 없는 값이라 제거
train_x = train.drop(columns=["ID", "Calories_Burned"],axis=1)
train_y = train["Calories_Burned"]
test_x = test.drop(columns=["ID"],axis=1)

###✳️ Data Preprocessing / 데이터 전처리

In [ ]:
# 5-1. Weight_Status 인코딩 (Ordinal)
# 👉 범주형 데이터를 숫자로 변환
status_map = {
    "Normal Weight": 0,
    "Overweight": 1,
    "Obese": 2
}
train_x["Weight_Status"] = train_x["Weight_Status"].map(status_map)
test_x["Weight_Status"] = test_x["Weight_Status"].map(status_map)

# 5-2. Gender 인코딩
# 👉 범주형 데이터를 숫자로 변환
le = LabelEncoder()
train_x["Gender"] = le.fit_transform(train_x["Gender"])
test_x["Gender"] = le.transform(test_x["Gender"])

# 5-3. train / validation split
# 👉 모델 성능을 확인하기 위해 학습/검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

# 5-4. Polynomial Feature
# 👉 변수 간 상호작용 추가
poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)

###✳️ Regression Model Definition / 회귀 모델 정의

In [ ]:
# 👉 선형 모델 + 정규화 모델(Ridge) 조합
# 👉 Ridge는 과적합 방지 역할
lr = LinearRegression()
ridge = Ridge(alpha=0.1)

###✳️ Model fitting / 모델 학습

In [ ]:
lr.fit(X_train_poly, y_train)
ridge.fit(X_train_poly, y_train)

###✳️ Inference / 추론

In [ ]:
# 8-1. Validation 예측
pred_lr_val = lr.predict(X_val_poly)
pred_ridge_val = ridge.predict(X_val_poly)

# 8-2. 앙상블 + 반올림
# 👉 두 모델 결과를 가중 평균 → 성능 향상
val_preds = 0.6 * pred_lr_val + 0.4 * pred_ridge_val
final_val_preds = np.round(val_preds)

# 8-3. RMSE 계산
rmse = np.sqrt(mean_squared_error(y_val, final_val_preds))
print(f"🔥 Validation RMSE: {rmse:.5f}")

###✳️ submit / 제출

In [ ]:
# 9-1. 전체 데이터로 재학습
full_train_poly = poly.fit_transform(train_x)
test_poly = poly.transform(test_x)

lr.fit(full_train_poly, train_y)
ridge.fit(full_train_poly, train_y)

# 9-2. 테스트 예측
final_preds = np.round(
    0.6 * lr.predict(test_poly) + 0.4 * ridge.predict(test_poly)
)

# 9-3. submission 생성
submission["Calories_Burned"] = final_preds

# 9-4. 저장
submission.to_csv("submission.csv", index=False)
print("✅ 제출 파일 생성 완료!")

# [두번째 제출]
“Polynomial Feature로 변수 간 상호작용을 반영해 선형 모델의 성능을 개선”

###✳️ Data Preprocessing / 데이터 전처리

In [ ]:
# 인코딩
status_map = {
    "Normal Weight": 0,
    "Overweight": 1,
    "Obese": 2
}

train_x["Weight_Status"] = train_x["Weight_Status"].map(status_map)
test_x["Weight_Status"] = test_x["Weight_Status"].map(status_map)

# 인코딩
le = LabelEncoder()
train_x["Gender"] = le.fit_transform(train_x["Gender"])
test_x["Gender"] = le.transform(test_x["Gender"])



# Feature Selection (핵심🔥)
# 👉 성능 개선을 위해 일부 컬럼 제거
# 👉 불필요하거나 노이즈가 되는 변수 제거 전략
drop_cols = [
    "Body_Temperature(F)",
    "Weight_Status",
    "Height(Feet)",
    "Height(Remainder_Inches)"
]

train_x = train_x.drop(columns=drop_cols)
test_x = test_x.drop(columns=drop_cols)

# 👉 현재 사용되는 피처
# ['Exercise_Duration', 'BPM', 'Weight(lb)', 'Gender', 'Age']

###✳️ Regression Model Definition / 회귀 모델 정의

In [ ]:
# 👉 Polynomial Feature 설정
# 👉 변수 간 상호작용 추가 → 선형 모델 성능 강화
poly = PolynomialFeatures(
    degree=3,
    interaction_only=True,
    include_bias=False
)

# 👉 기본 선형 모델 + 규제 모델(Ridge)
# 👉 Ridge는 과적합 방지 역할
lr = LinearRegression()
ridge = Ridge(alpha=0.004)  # 👉 튜닝된 값

###✳️ Model fitting / 모델 학습

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_x, train_y, test_size=0.2, random_state=777  # 👉 성능 좋았던 seed
)

# 👉 Polynomial Feature 적용
X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)

# 👉 모델 학습
lr.fit(X_train_poly, y_train)
ridge.fit(X_train_poly, y_train)

# 👉 예측 수행
pred_lr = lr.predict(X_val_poly)
pred_ridge = ridge.predict(X_val_poly)

# 👉 앙상블 (가중 평균 + 반올림)
val_pred = 0.45 * np.round(pred_lr) + 0.55 * np.round(pred_ridge)

rmse = np.sqrt(mean_squared_error(y_val, val_pred))

print("🔥 RMSE:", rmse)

###✳️ Inference / 추론

In [ ]:
# 👉 테스트 데이터에도 동일하게 Polynomial 적용
test_x_poly = poly.transform(test_x)

# 👉 각 모델 예측
pred_lr = lr.predict(test_x_poly)
pred_ridge = ridge.predict(test_x_poly)

# 👉 최종 앙상블 (검증과 동일한 방식 유지)
final_pred = 0.45 * np.round(pred_lr) + 0.55 * np.round(pred_ridge)

###✳️ submit / 제출

In [ ]:
# 👉 제출 파일에 예측값 삽입
submission["Calories_Burned"] = final_pred

# 👉 CSV 파일로 저장
submission.to_csv("final_0097.csv", index=False)

print("🔥 제출 완료: final_0097.csv")

# [세번째 제출]
Keytel 공식 기반 칼로리 raw 예측 + catboost를 통한
잔차 학습

In [ ]:
# ============================================================
# Cell 1 : import / 라이브러리 호출
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("catboost")
install("optuna")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings("ignore")

print("✅ 라이브러리 로드 완료")


# ============================================================
# Cell 2 : Fixed RandomSeed / 랜덤시드 고정
# ============================================================
SEED = 42
np.random.seed(SEED)
print("✅ 랜덤시드 고정 완료 (SEED =", SEED, ")")


# ============================================================
# Cell 3 : Data Load / 데이터 불러오기
# ============================================================
# Google Drive 마운트 사용 시 아래 주석 해제 후 경로 수정
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/your_folder/"
DATA_DIR = "/content/"

train = pd.read_csv(DATA_DIR + "train.csv")
test  = pd.read_csv(DATA_DIR + "test.csv")

print("✅ 데이터 로드 완료")
print("   Train shape:", train.shape)
print("   Test  shape:", test.shape)
train.head(3)


# ============================================================
# Cell 4 : Feature & Target Split / 독립변수, 종속변수로 나누기
# ============================================================
TARGET    = "Calories_Burned"
DROP_COLS = ["ID", TARGET]

train_x = train.drop(columns=DROP_COLS).copy()
train_y = train[TARGET].copy()
test_x  = test.drop(columns=["ID"], errors="ignore").copy()

print("✅ Feature / Target 분리 완료")
print("   train_x:", train_x.shape, "| train_y:", train_y.shape)
print("   test_x :", test_x.shape)


# ============================================================
# Cell 5 : Data Preprocessing / 데이터 전처리
# ============================================================

# ── 범주형 인코딩 (전처리 먼저)
status_map = {"Normal Weight": 0, "Overweight": 1, "Obese": 2}

for df in [train_x, test_x]:
    df["Weight_Status"] = df["Weight_Status"].map(status_map)

le = LabelEncoder()
le.fit(train_x["Gender"])
train_x["Gender"] = le.transform(train_x["Gender"])
test_x["Gender"]  = le.transform(test_x["Gender"])

# Gender 인코딩 결과 확인 (Keytel 공식에서 남성 판별 기준)
print("Gender 인코딩 매핑:", dict(zip(le.classes_, le.transform(le.classes_))))
# → 예: {'F': 0, 'M': 1}  이면 Male == 1 이 맞음
MALE_CODE = int(le.transform(["M"])[0])
print(f"   남성(M) 코드: {MALE_CODE}  ← Keytel 공식에서 이 값으로 남/여 분기")


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """피처 엔지니어링 (train / test 공통 적용)"""
    df = df.copy()

    # 1. Height 통합
    df["Height_inches"] = df["Height(Feet)"] * 12 + df["Height(Remainder_Inches)"]
    df["Height_m"]      = df["Height_inches"] * 0.0254

    # 2. 단위 변환
    df["Weight_kg"] = df["Weight(lb)"] / 2.2046
    df["Temp_C"]    = (df["Body_Temperature(F)"] - 32) * 5 / 9

    # 3. BMI
    df["BMI"]     = df["Weight_kg"] / (df["Height_m"] ** 2)
    df["BMI_Age"] = df["BMI"] * df["Age"]

    # 4. 운동 강도
    df["Intensity"]        = df["BPM"] * df["Exercise_Duration"]
    df["Intensity_Weight"] = df["Intensity"] * df["Weight_kg"]

    # 5. Body Temperature 복합 피처
    df["Temp_BPM"]      = df["Temp_C"] * df["BPM"]
    df["Temp_Duration"] = df["Temp_C"] * df["Exercise_Duration"]
    df["Temp_Weight"]   = df["Temp_C"] * df["Weight_kg"]

    # 6. 나이 관련
    df["Age_BPM"]     = df["Age"] * df["BPM"]
    df["Age_squared"] = df["Age"] ** 2

    return df


train_x = add_features(train_x)
test_x  = add_features(test_x)

print("✅ 전처리 완료")
print("   최종 피처 수:", train_x.shape[1])


# ============================================================
# Cell 6 : Regression Model Definition / 회귀 모델 정의
# ============================================================

# ── 기존 검증된 Keytel 파라미터 (고정)
best_params = [
    2.01715651e-01,   # 남 age
    4.09789570e-02,   # 남 weight(lb)
    6.30933782e-01,   # 남 bpm
   -5.50993643e+01,   # 남 const
    7.40152512e-02,   # 여 age
   -5.74001235e-02,   # 여 weight(kg)
    4.47193226e-01,   # 여 bpm
   -2.04026456e+01,   # 여 const
]


def predict_with_params_raw(df: pd.DataFrame, params: list) -> np.ndarray:
    """
    Keytel 공식 기반 칼로리 raw 예측
    - df: 인코딩 완료된 DataFrame (Gender가 이미 0/1)
    - MALE_CODE 기준으로 남/여 분기
    """
    df = df.copy()
    (m_age, m_wlb, m_bpm, m_const,
     f_age, f_wkg, f_bpm, f_const) = params

    weight_kg = df["Weight(lb)"] / 2.2046

    cal_m = (
          df["Age"]        * m_age
        + df["Weight(lb)"] * m_wlb
        + df["BPM"]        * m_bpm
        + m_const
    ) * df["Exercise_Duration"] / 4.184

    cal_f = (
          df["Age"]  * f_age
        + weight_kg  * f_wkg
        + df["BPM"]  * f_bpm
        + f_const
    ) * df["Exercise_Duration"] / 4.184

    # MALE_CODE 로 남/여 분기 (인코딩 결과에 따라 자동 적용)
    return np.where(df["Gender"] == MALE_CODE, cal_m, cal_f)


def finalize_pred(raw_pred: np.ndarray) -> np.ndarray:
    pred = np.floor(raw_pred + 0.5)
    pred = np.clip(pred, 0, None)
    return pred


print("✅ 모델 정의 완료")


# ============================================================
# Cell 7 : Model fitting / 모델 학습
# ============================================================

# ── 이상치 제거 및 Validation Split
villain_idx = [4458, 231]

base_x = train_x.drop(index=villain_idx, errors="ignore").copy()
base_y = train_y.drop(index=villain_idx, errors="ignore").copy()

X_train, X_val, y_train, y_val = train_test_split(
    base_x, base_y, test_size=0.2, random_state=SEED
)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")

# ── Keytel 공식 예측 & Residual 계산 (인코딩된 train_x 그대로 사용)
train_raw      = predict_with_params_raw(X_train, best_params)
train_pred     = finalize_pred(train_raw)
train_residual = y_train - train_pred

keytel_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
print(f"Keytel 공식 Train RMSE: {keytel_rmse:.4f}  (← 기존처럼 낮으면 정상)")

# ──────────────────────────────────────────────────────────
# CatBoost 하이퍼파라미터 Optuna 최적화
# CATBOOST_TUNE = False 로 바꾸면 기본값 사용
# ──────────────────────────────────────────────────────────
CATBOOST_TUNE = True
OPTUNA_TRIALS = 100

if CATBOOST_TUNE:
    print(f"\n🔍 CatBoost 하이퍼파라미터 탐색 시작 ({OPTUNA_TRIALS} trials)...")

    def cb_objective(trial):
        params = {
            "iterations":       trial.suggest_int  ("iterations",       200, 1000),
            "depth":            trial.suggest_int  ("depth",            4,   8),
            "learning_rate":    trial.suggest_float("learning_rate",    0.01, 0.15),
            "l2_leaf_reg":      trial.suggest_float("l2_leaf_reg",      1.0,  10.0),
            "min_data_in_leaf": trial.suggest_int  ("min_data_in_leaf", 1,    50),
            "random_state":     SEED,
            "verbose":          0,
        }
        model = CatBoostRegressor(**params)
        model.fit(X_train, train_residual)

        val_raw_      = predict_with_params_raw(X_val, best_params)
        val_residual_ = model.predict(X_val)
        val_final_    = finalize_pred(val_raw_ + val_residual_)

        return np.sqrt(mean_squared_error(y_val, val_final_))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(cb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best_cb_params = study.best_params
    best_cb_params["random_state"] = SEED
    best_cb_params["verbose"]      = 0
    print(f"✅ CatBoost 튜닝 완료 | 최적 Val RMSE: {study.best_value:.6f}")
    print(f"   최적 파라미터: {best_cb_params}")

else:
    best_cb_params = {
        "iterations":   500,
        "depth":        5,
        "learning_rate": 0.05,
        "random_state": SEED,
        "verbose":      0,
    }
    print("⏩ CatBoost 튜닝 건너뜀 → 기본 파라미터 사용")

# ── 최적 파라미터로 최종 모델 재학습
print("\n🚀 최종 CatBoost 모델 학습 중...")
res_model = CatBoostRegressor(**best_cb_params)
res_model.fit(X_train, train_residual)
print("✅ 모델 학습 완료")


# ============================================================
# Cell 8 : Inference / 추론
# ============================================================

# ── Validation 성능 확인
val_raw      = predict_with_params_raw(X_val, best_params)
val_residual = res_model.predict(X_val)
val_final    = finalize_pred(val_raw + val_residual)

rmse = np.sqrt(mean_squared_error(y_val, val_final))
print(f"✅ Validation RMSE: {rmse:.6f}")

# ── Test 예측
test_raw      = predict_with_params_raw(test_x, best_params)
test_residual = res_model.predict(test_x)
test_final    = finalize_pred(test_raw + test_residual)

print(f"✅ Test 예측 완료 | 예측값 샘플: {test_final[:5]}")


# ============================================================
# Cell 9 : Submit / 제출
# ============================================================
submission = pd.read_csv(DATA_DIR + "sample_submission.csv")
submission[TARGET] = test_final
submission.to_csv("submission.csv", index=False)

print("✅ submission.csv 저장 완료")

# Colab 파일 자동 다운로드 원하면 아래 주석 해제
# from google.colab import files
# files.download("submission.csv")

submission.head()

# [미제출]
마감시간에 맞춰 제출하지 못했지만
시도를 지속한 끝에 여기까지 해봤습니다.

데이터 전처리까지 동일


In [ ]:
# 수많은 시도 끝 결국 가장 많이 문제가 많이 발생한 구간의 상호작용 특성을 추가
def clean_and_fe(df):

    # 4. 특성 추가: Exercise_Duration과 Age를 묶음.
    df['E_A_I'] = df['Exercise_Duration'] * df['Age']

    # 5. 특성 추가: BPM과 Weight(lb)를 묶음
    df['B_W_I'] = df['BPM'] * df['Weight(lb)']

    # 6. 특성 추가: BPM과 Age를 묶음
    df['B_A_I'] = df['BPM'] * df['Age']

    return df

In [ ]:
# 적용 및 확인
train_x = clean_and_fe(train_x)
test_x = clean_and_fe(test_x)

print("--- [현제 컬럼] ---")
print(train_x.columns)

In [ ]:
# [Step 2] Train / Val 분리 (8:2 비율)
from sklearn.model_selection import train_test_split, cross_val_score
# 이 시점에서 검증 데이터가 따로 떨어져 나와야 신뢰할 수 있는 점수가 나옵니다.
X_train, val_x, y_train, y_val = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

In [ ]:
#L1(Lasso) + L2(Ridge) 규제를 함께 적용하며, 교차 검증을 통해 최적의 alpha와 l1_ratio를 자동으로 찾는 ElasticNetCV를 적용
from sklearn.linear_model import ElasticNetCV

#인터넷 검색으로 자주발견된 칼로리 계산공식 반영
def apply_keytel(df):
    w_kg = df['Weight(lb)'] / 2.2046
    age = df['Age']
    bpm = df['BPM']
    dur = df['Exercise_Duration']

    cal_male = (((age * 0.2017) + (w_kg * 0.09036) + (bpm * 0.6309) - 55.0969) * dur / 4.184).round(2)
    cal_female = (((age * 0.074) - (w_kg * 0.05741) + (bpm * 0.4472) - 20.4022) * dur / 4.184).round(0)

    return np.where(df['Gender'] == 1, cal_male, cal_female)


# 잔차 모델을 위한 특성 정의
features_for_res = ['Keytel_Pred','E_A_I','B_W_I','B_A_I']

# --- 가장 좋은 성능을 보이는 설정 적용 ---
best_scaler_name = 'StandardScaler'
best_alpha = 100 # ElasticNetCV가 최적 alpha를 찾습니다.

print(f"\n--- 최적 조합 적용: Scaler: {best_scaler_name}, Model: ElasticNetCV ---")

# 처리를 위해 X_train과 val_x의 복사본 생성
X_train_processed = X_train.copy()
val_x_processed = val_x.copy()

# 훈련 및 검증 세트에 대해 Keytel_Pred 계산
X_train_processed['Keytel_Pred'] = apply_keytel(X_train_processed)
val_x_processed['Keytel_Pred'] = apply_keytel(val_x_processed)

# 잔차 모델 훈련을 위한 잔차 계산
y_residual = y_train - X_train_processed['Keytel_Pred']

# 잔차 모델을 위한 특성 추출
X_train_features_for_res = X_train_processed[features_for_res].copy()
val_x_features_for_res = val_x_processed[features_for_res].copy()

# StandardScaler 적용
scaler = StandardScaler()
scaler.fit(X_train_features_for_res)
X_train_features_for_res_scaled = scaler.transform(X_train_features_for_res)
val_x_features_for_res_scaled = scaler.transform(val_x_features_for_res)

# 최적 alpha 및 l1_ratio를 찾는 ElasticNetCV 모델 훈련
model_res = ElasticNetCV(cv=5, random_state=42, n_jobs=-1) # cv=5, n_jobs=-1 추가
model_res.fit(X_train_features_for_res_scaled, y_residual)

# 검증 세트에 대한 예측 수행
val_base_pred = val_x_processed['Keytel_Pred']
val_residual_pred = model_res.predict(val_x_features_for_res_scaled)

# 기본 및 잔차 예측 결합
val_final_pred = val_base_pred + val_residual_pred

# 후처리
val_final_round = np.round(val_final_pred.clip(lower=0))

# RMSE 평가
rmse_final = np.sqrt(mean_squared_error(y_val, val_final_round))
print(f"최종 RMSE (최적 모델): {rmse_final:.5f}")
print(f"ElasticNetCV 최적 alpha: {model_res.alpha_:.5f}")
print(f"ElasticNetCV 최적 l1_ratio: {model_res.l1_ratio_:.5f}")

In [ ]:
# 테스트 데이터에 대한 Keytel_Pred 계산
test_x_processed = test_x.copy()
test_x_processed['Keytel_Pred'] = apply_keytel(test_x_processed)

# 잔차 모델을 위한 특성 추출 및 스케일링
test_x_features_for_res = test_x_processed[features_for_res].copy()
test_x_features_for_res_scaled = scaler.transform(test_x_features_for_res)

# 테스트 세트에 대한 예측 수행
test_base_pred = test_x_processed['Keytel_Pred']
test_residual_pred = model_res.predict(test_x_features_for_res_scaled)

# 기본 및 잔차 예측 결합
test_final_pred = test_base_pred + test_residual_pred

# 후처리 (음수 값 방지 및 반올림)
test_final_round = np.round(test_final_pred.clip(lower=0))

# submission 파일 생성
submission['Calories_Burned'] = test_final_round

# submission.csv 파일 저장
submission.to_csv('submission.csv', index=False)

print("submission.csv 파일이 성공적으로 생성되었습니다.")